# Análisis del DATASET

In [1]:
import pandas as pd
import numpy as np
import os

# ---------------------------------------------------------
# BLOQUE 1: Configuración de Rutas y Carga Inicial
# ---------------------------------------------------------
folder_path = r'../data/raw' 
path_main = os.path.join(folder_path, 'all_stocks_5yr.csv')

print("--- BLOQUE 1: Carga y Ordenamiento ---")
print("Cargando dataset principal...")
# df = pd.read_csv(path_main) # Descomenta esta línea en tu entorno local

# Simulación de carga para demostración en Colab
df = pd.read_csv(path_main) 

# 1. Convertir la fecha a formato datetime
df['date'] = pd.to_datetime(df['date'])

# 2. Asegurar el orden cronológico
df = df.sort_values(['Name', 'date']).reset_index(drop=True)
print(f"Dataset cargado y ordenado cronológicamente.")
print(f"Total de filas iniciales: {len(df)}")
print(f"Total de acciones únicas: {df['Name'].nunique()}\n")

--- BLOQUE 1: Carga y Ordenamiento ---
Cargando dataset principal...
Dataset cargado y ordenado cronológicamente.
Total de filas iniciales: 619040
Total de acciones únicas: 505



In [2]:
# ---------------------------------------------------------
# BLOQUE 2: Cálculo de Retornos Diarios y Auditoría de NaNs
# ---------------------------------------------------------
print("--- BLOQUE 2: Cálculo de Retornos Diarios ---")

# 3. Calcular el retorno diario usando la fórmula (P_t / P_{t-1}) - 1
print("Calculando el cambio porcentual agrupado por cada acción ('Name')...")
df['return_1d'] = df.groupby('Name')['adjusted_price'].pct_change()

# Auditoría de los valores nulos (NaN) generados
nan_count_initial = df['return_1d'].isna().sum()
unique_stocks = df['Name'].nunique()

print(f"-> Se generaron {nan_count_initial} valores NaN en la columna 'return_1d'.")
print(f"-> Esperábamos al menos {unique_stocks} NaNs (uno por el primer día de cada acción).")

if nan_count_initial > unique_stocks:
    print(f"-> ¡Nota! Hay {nan_count_initial - unique_stocks} NaNs adicionales. Esto suele ocurrir si hay saltos o días faltantes intermedios en el histórico de algunas acciones.")
elif nan_count_initial == unique_stocks:
    print("-> ¡Perfecto! La cantidad de NaNs coincide exactamente con el número de acciones. No hay saltos intermedios.")

# 4. Limpieza: Eliminar los registros que no tienen retorno
print("\nRemoviendo los registros con valores NaN en el retorno...")
df_limpio = df.dropna(subset=['return_1d']).reset_index(drop=True)

# Verificación final
nan_count_final = df_limpio['return_1d'].isna().sum()
filas_removidas = len(df) - len(df_limpio)

print(f"-> Valores NaN restantes: {nan_count_final}")
print(f"-> Total de filas removidas del dataset: {filas_removidas}")
print(f"-> Total de registros finales listos para el análisis: {len(df_limpio)}\n")

# Actualizamos el dataframe principal
df = df_limpio.copy()

--- BLOQUE 2: Cálculo de Retornos Diarios ---
Calculando el cambio porcentual agrupado por cada acción ('Name')...
-> Se generaron 505 valores NaN en la columna 'return_1d'.
-> Esperábamos al menos 505 NaNs (uno por el primer día de cada acción).
-> ¡Perfecto! La cantidad de NaNs coincide exactamente con el número de acciones. No hay saltos intermedios.

Removiendo los registros con valores NaN en el retorno...
-> Valores NaN restantes: 0
-> Total de filas removidas del dataset: 505
-> Total de registros finales listos para el análisis: 618535



In [5]:
df.to_csv("../data/processed/dataset_with_returns.csv", index=False)